# Setup

This problem studies the computability of the exponential function.

- `Rational` denotes the type of rational numbers (`fractions.Fraction`).
- `CReal` represents a real $x$ by rational approximations $q_n\in\mathbb{Q}$ satisfying $|q_n-x|<1/2^n$ for every $n\geq0$.
- `x(n)` returns the rational approximation $q_n$.
- Your function takes a `CReal` object and returns a new `CReal` object.

Notes.

- Your implementation must work for every valid input sequence. Access an input real only through `x(n)`.
- Perform computations using integers and rational arithmetic. Construct rational constants as `Rational(a, b)`; do not use floating-point arithmetic or numerical libraries.

In [1]:
import typing, fractions, decimal

In [2]:
Rational = fractions.Fraction

def require_natural(value: int, name: str = "n") -> None:
    assert(type(value) is int and value >= 0), f"{name} must be a nonnegative integer"


def radius(n: int) -> Rational:
    # Return 1/2^n exactly.
    require_natural(n)
    return Rational(1, 1 << n)


class CReal:
    # A real x, given by rational approximations with error < 1/2^n.
    def __init__(self, approximation: typing.Callable[[int], Rational]):
        assert(callable(approximation))
        self._approximation = approximation
        self._cache = {}

    def __call__(self, n: int) -> Rational:
        # Compute each requested approximation once and cache its value.
        require_natural(n)
        if n not in self._cache:
            q = self._approximation(n)
            assert(isinstance(q, Rational)), "Every approximation must be a Rational"
            self._cache[n] = q
        return self._cache[n]


def enclosure(x: CReal, n: int) -> tuple[Rational, Rational]:
    # Return endpoints of an interval containing x strictly inside.
    q = x(n)
    return q - radius(n), q + radius(n)


def decimal_preview(q: Rational, digits: int = 12) -> str:
    # Return a rounded decimal string for display.
    with decimal.localcontext() as ctx:
        ctx.prec = digits
        return str(decimal.Decimal(q.numerator) / decimal.Decimal(q.denominator))

The following functions are provided for testing. You don't need to understand their implementations.

- `rational_test_input(a, mode)` represents a rational $a$ by either $q_n=a$ (`"exact"`) or $q_n=a+(-1)^n\frac{15}{16}\frac{1}{2^n}$ (`"alternating"`). Both satisfy the same error bound.
- `decimal_preview()` is for display only. It rounds a rational approximation; it does not certify correctly rounded digits of the represented real.

In [3]:
def rational_test_input(a: Rational, mode: str = "alternating") -> CReal:
    if not isinstance(a, Rational):
        raise TypeError("Use Rational for the test value")
    if mode == "exact":
        return CReal(lambda n: a)
    if mode == "alternating":
        return CReal(
            lambda n: a + (-1 if n % 2 else 1) * Rational(15, 16) * radius(n)
        )
    raise ValueError("Unknown test-input mode")


# Example
demo = rational_test_input(Rational(1, 3))
for n in [0, 2, 5]:
    lo, hi = enclosure(demo, n)
    print(f"n={n}: q={demo(n)}, interval=[{lo}, {hi}]")

n=0: q=61/48, interval=[13/48, 109/48]
n=2: q=109/192, interval=[61/192, 157/192]
n=5: q=467/1536, interval=[419/1536, 515/1536]


# Problem 3(b)

Implement the function `creal_exp()`.

- `creal_exp(x)` takes a `CReal` representing any real $x$ and returns a `CReal` representing $\exp(x)$.

The returned object `y` must return a `Rational` satisfying $|y(n)-\exp(x)|<1/2^n$ for every integer $n\geq0$. This must hold for every valid input approximation sequence.

In [4]:
def creal_exp(x: CReal) -> CReal:
    assert(isinstance(x, CReal))

    def approximation(n: int) -> Rational:
        # A computable integer B bounds both |x| and every queried |x(m)|.
        q0 = x(0)
        B = 3
        while Rational(B) <= abs(q0) + 2:
            B += 1

        # exp(t) <= exp(B) < 3^B on [-B, B].
        C = Rational(1)
        for _ in range(B):
            C *= 3
        budget = radius(n + 2)

        # Bound the sensitivity to the error in the rational input.
        m = 0
        while C * radius(m) >= budget:
            m += 1
        q = x(m)

        # Bound the Taylor remainder C*B^(d+1)/(d+1)!.
        k = 1
        tail_bound = C * B
        while tail_bound >= budget:
            k += 1
            tail_bound = tail_bound * B / k
        d = k - 1

        # All intermediate values are exact fractions.
        term = Rational(1)
        total = term
        for j in range(1, d + 1):
            term = term * q / j
            total += term
        return total

    return CReal(approximation)


In [5]:
# Simple tests
for q in [Rational(-1), Rational(0), Rational(1)]:
    x = rational_test_input(q)
    print( f"exp({q}) ~", decimal_preview(creal_exp(x)(10)) )

exp(-1) ~ 0.367876809902
exp(0) ~ 0.999992847468
exp(1) ~ 2.71827696780
